In [166]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

# from lesson_1.ingest import load_faq_data
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np
from elasticsearch import Elasticsearch

In [196]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Q1. Embedding a query

In [6]:
query = "How does approximate nearest neighbor search work?"
v_query = model.encode(query)
v_query[0]

np.float32(-0.020581992)

### Q2. Cosine similarity


In [168]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [12]:
documents[22]

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [9]:
target_filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'
for i, doc in enumerate(documents):
    if doc['filename'] == target_filename:
        target_content = doc['content']
        target_index = i
target_index, target_content

(22,
 '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far

In [10]:
embedded_target_content = model.encode(target_content)
embedded_target_content.dot(v_query)

np.float32(0.45681334)

### Q3. Chunking and search by hand

In [169]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks), len(documents[71]['content'])

(295, 2348)

In [35]:
chunks[22:24]

[{'start': 1000,
  'content': 'to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n"""\n```\n\nThis is what grounds the answer in our data and reduces hallucinations.\n\n## The user prompt template\n\nThe user prompt template has placeholders for the question and the\ncontext:\n\n```python\nUSER_PROMPT_TEMPLATE = """\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\n## Building the context\n\nThe `context` is a formatted string with all the search results:\n\n```python\ndef build_context(search_results):\n    lines = []\n\n    for doc in search_results:\n        lines.append(doc["section"])\n        lines.append("Q: " + doc["question"])\n        lines.append("A: " + doc["answer"])\n        lines.append("")\n\n    return "\\n".join(lines).strip()\n```\n\nEach document becomes a block with the section, question, and answer.\nThis format makes it easy for the LLM to read. We turned a list of\ndicti

In [170]:
embedded_chunks = []
for chunk in chunks:
    e_c = model.encode(chunk['content'])
    embedded_chunks.append(e_c)
len(embedded_chunks), len(embedded_chunks[22])

(295, 384)

In [171]:
X = np.array(embedded_chunks)
X.shape

(295, 384)

In [193]:
scores = X.dot(v_query)

In [54]:
idx = np.argmax(scores)
scores[idx], chunks[idx]

(np.float32(0.5624412),
 {'start': 1000,
  'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because th

### Q4. Vector search with ES

In [197]:
es = Elasticsearch("http://localhost:9200")

In [181]:
# Delete first in case the index existed:
es.indices.delete(index="chunk_homework_2_lessons", ignore_unavailable=True)

# Create index:
index_settings = {
    "mappings": {
        "properties": {
            "filename": {"type": "keyword"}
            ,"content": {"type": "text"}
            ,"vector": {
                "type": "dense_vector"
                ,"dims": len(X[0])
                ,"index": True
                ,"similarity": "cosine"
            }
        }
    }
}

es.indices.create(index="chunk_homework_2_lessons", body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'chunk_homework_2_lessons'})

In [180]:
len(embedded_chunks)

295

In [182]:
for i, chunk in enumerate(embedded_chunks):
    es.index(
        index="chunk_homework_2_lessons",
        id=i,
        body={
            "filename": chunks[i]["filename"]
            ,"content": chunks[i]["content"]
            ,"vector": X[i].tolist()
        }
    )

es.count(index="chunk_homework_2_lessons")['count']


295

In [185]:
query_2 = 'What metric do we use to evaluate a search engine?'
v_query_2 = model.encode(query_2)
# v_query_2

In [ ]:
# RFF score : can you explain briefly the formula and give me sample example. I have here 

In [233]:
def vector_search(query):
    response = es.search(
        index="chunk_homework_2_lessons"
        ,body={
            "knn": {
                "field": "vector"
                ,"query_vector": query.tolist()
                ,"k": 10
                ,"num_candidates": 50
            }
        }
    )

    results = {
        hit["_source"]["filename"]: hit["_score"]
        for hit in response["hits"]["hits"]
    }
    return dict(sorted(results.items(), key=lambda x: x[1], reverse=True))



In [220]:
vector_search(v_query_2)

{'04-evaluation/lessons/01-intro.md': 0.7892268,
 '04-evaluation/lessons/05-search-metrics.md': 0.7669859,
 '01-agentic-rag/lessons/05-search.md': 0.74463177}

### Q5. Text search vs Vector search

In [221]:
def text_search(query):
    fields = ["content"]
    results = es.search(
        index="chunk_homework_1_lessons"
        ,body={
            "size": 5
            ,"query": {
                "bool": {
                    "must": {"multi_match": {"query": query, "fields": fields}}
                }
            }
        }
    )

    return {
        hit["_source"]["filename"]: hit["_score"]
        for hit in results["hits"]["hits"]
    }
    # 

In [222]:
query_3 = 'How do I store vectors in PostgreSQL?'
v_query_3 = model.encode(query_3)

In [242]:
vector_search_result = vector_search(v_query_3)
vector_search_filename = list(vector_search_result.keys())
vector_search_filename

['02-vector-search/lessons/02-embeddings.md',
 '05-monitoring/lessons/05-database.md',
 '02-vector-search/lessons/04-vector-search.md',
 '02-vector-search/lessons/08-pgvector.md']

In [243]:
text_search_result = text_search(query_3)
text_search_filename = list(text_search_result.keys())
text_search_filename

['02-vector-search/lessons/01-intro.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md']

In [244]:
[i for i in vector_search_result  if i not in text_search_result]

['02-vector-search/lessons/02-embeddings.md',
 '05-monitoring/lessons/05-database.md',
 '02-vector-search/lessons/04-vector-search.md']

### Q6. Hybrid Search

In [299]:
query_4 = "How do I give the model access to tools?"
v_query_4 = model.encode(query_4)


In [301]:
text_search_result = text_search(query_4)
text_search_result

{'02-vector-search/lessons/08-pgvector.md': 9.107528,
 '01-agentic-rag/lessons/13-function-calling.md': 8.18919,
 '01-agentic-rag/lessons/10-rag-next-steps.md': 8.486778}

In [302]:
vector_search_result = vector_search(v_query_4)
vector_search_result

{'05-monitoring/lessons/02-assistant-setup.md': 0.6813377,
 '01-agentic-rag/lessons/15-frameworks.md': 0.643176,
 '02-vector-search/lessons/02-embeddings.md': 0.64131117,
 '01-agentic-rag/lessons/01-intro.md': 0.63765705,
 '01-agentic-rag/lessons/08-rag-helper.md': 0.63450193,
 '01-agentic-rag/lessons/13-function-calling.md': 0.6335809,
 '01-agentic-rag/lessons/14-agentic-loop.md': 0.63217413}

In [303]:
vector_ranked = sorted(vector_search_result, key=vector_search_result.get, reverse=True)
rrf_scores = {}
k = 60
for rank, doc in enumerate(vector_ranked, start=1):
    rrf_scores[doc] = rrf_scores.get(doc, 0) + 1/(k + rank)
rrf_scores

{'05-monitoring/lessons/02-assistant-setup.md': 0.01639344262295082,
 '01-agentic-rag/lessons/15-frameworks.md': 0.016129032258064516,
 '02-vector-search/lessons/02-embeddings.md': 0.015873015873015872,
 '01-agentic-rag/lessons/01-intro.md': 0.015625,
 '01-agentic-rag/lessons/08-rag-helper.md': 0.015384615384615385,
 '01-agentic-rag/lessons/13-function-calling.md': 0.015151515151515152,
 '01-agentic-rag/lessons/14-agentic-loop.md': 0.014925373134328358}

In [304]:
text_ranked = sorted(text_search_result, key = text_search_result.get, reverse=True)
text_ranked
for rank, doc in enumerate(text_ranked, start = 1):
    rrf_scores[doc] = rrf_scores.get(doc,0) + 1/(k + rank)
rrf_scores

{'05-monitoring/lessons/02-assistant-setup.md': 0.01639344262295082,
 '01-agentic-rag/lessons/15-frameworks.md': 0.016129032258064516,
 '02-vector-search/lessons/02-embeddings.md': 0.015873015873015872,
 '01-agentic-rag/lessons/01-intro.md': 0.015625,
 '01-agentic-rag/lessons/08-rag-helper.md': 0.015384615384615385,
 '01-agentic-rag/lessons/13-function-calling.md': 0.031024531024531024,
 '01-agentic-rag/lessons/14-agentic-loop.md': 0.014925373134328358,
 '02-vector-search/lessons/08-pgvector.md': 0.01639344262295082,
 '01-agentic-rag/lessons/10-rag-next-steps.md': 0.016129032258064516}

In [307]:
sorted(rrf_scores, key = rrf_scores.get, reverse=True)

['01-agentic-rag/lessons/13-function-calling.md',
 '05-monitoring/lessons/02-assistant-setup.md',
 '02-vector-search/lessons/08-pgvector.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/10-rag-next-steps.md',
 '02-vector-search/lessons/02-embeddings.md',
 '01-agentic-rag/lessons/01-intro.md',
 '01-agentic-rag/lessons/08-rag-helper.md',
 '01-agentic-rag/lessons/14-agentic-loop.md']

### EXTRA: FILTER by filename to extract vectors of those files


In [279]:
text_search_filename

['02-vector-search/lessons/01-intro.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md']

In [313]:

response = es.search(
    index="chunk_homework_2_lessons"
    ,body={
        "query": {
            "terms": {
                "filename":text_search_filename
                }
            }
        ,"_source": ["filename", "vector"]
        ,"size": 100
    }
)

result = []
for hit in response["hits"]["hits"]:
    result.append(hit["_source"])

result
# list(dict.fromkeys(result))

[{'filename': '02-vector-search/lessons/01-intro.md',
  'vector': [-0.03177938610315323,
   -0.00738902622833848,
   -0.012285708449780941,
   0.00012476372648961842,
   0.04801041632890701,
   -0.003842672798782587,
   0.008821690455079079,
   -0.0183199942111969,
   -0.039364155381917953,
   -0.023302610963582993,
   0.010226179845631123,
   0.03953111916780472,
   0.08086343854665756,
   -0.025984158739447594,
   -0.0038829660043120384,
   -0.06523920595645905,
   -0.018271563574671745,
   0.0832456648349762,
   0.00869089923799038,
   -0.015203423798084259,
   -0.012633057311177254,
   0.029979417100548744,
   -0.017764516174793243,
   -0.03709680214524269,
   0.021148191764950752,
   0.05958086997270584,
   0.0034374017268419266,
   -0.07958385348320007,
   0.003078251611441374,
   0.01751762256026268,
   0.0451345220208168,
   0.10791190713644028,
   0.04380190372467041,
   0.15410923957824707,
   -0.03319641202688217,
   0.03588919714093208,
   -0.0756511464715004,
   0.01035440